## Préparation

### Exercice 1

In [68]:
import numpy as np 
import cv2 
from plotly import express as px 
import msicpe 
import msicpe.tsi as tsi

In [69]:
im = cv2.imread('bloodcells.png',0)
px.imshow(im, title="Bloodcells", color_continuous_scale='gray').show()
h, w = im.shape

In [70]:
def kmeans_segmentation(im, K, max_iter=100):
    pixels = im.flatten().astype(float)

    means = np.random.choice(pixels, K, replace=False)
    
    for _ in range(max_iter):
        distances = np.abs(pixels[:, np.newaxis] - means)
        labels = np.argmin(distances, axis=1)
        new_means = np.array([pixels[labels == k].mean() if np.any(labels == k) else means[k] for k in range(K)])
        if np.all(means == new_means):
            break
        means = new_means
        
    return labels.reshape(im.shape), means

In [71]:
labels, means = kmeans_segmentation(im, K=2)

px.imshow(labels, title="Carte des segments (Labels)", color_continuous_scale='Viridis').show()

im_seg = means[labels]

px.imshow(im_seg, title="Image segmentée (Intensités moyennes)", color_continuous_scale='gray').show() 

## 1 Transformations d’histogramme et segmentation des globules

In [72]:
# 1. Lecture
im = cv2.imread('bloodcells.png', 0)

# 2. Segmentation K-means (K=2)
labels_init, means_init = kmeans_segmentation(im, K=2)

In [73]:
# 3. Histogramme normalisé et cumulé
H, bin_edges = np.histogram(im, bins=256, range=[0, 255])
H_norm = H / im.size  # Normalisation
Hc = np.cumsum(H_norm)  # Histogramme cumulé 

# Affichage 
centers = (bin_edges[:-1] + bin_edges[1:]) / 2
tsi.plotHistograms(centers, H_norm, Hc, title="Histogramme original")

In [74]:
# 4. Égalisation manuelle
# On crée une table de correspondance (LUT)
lut = np.round(255 * Hc).astype('uint8')

# Application à l'image
im_eq = lut[im]

# 5. Affichage des nouveaux histogrammes
H_eq, _ = np.histogram(im_eq, bins=256, range=[0, 255])
Hc_eq = np.cumsum(H_eq / im_eq.size)
tsi.plotHistograms(centers, H_eq / im_eq.size, Hc_eq, title="Histogramme égalisé")

px.imshow(im_eq, title="Image égalisée", color_continuous_scale='gray').show()

In [ ]:
# 6. Nouveau K-means sur l'image égalisée
labels_final, means_final = kmeans_segmentation(im_eq, K=2)

# Affichage du résultat binaire
im_bin = (labels_final == np.argmin(means_final)).astype('uint8')
px.imshow(im_bin, title="Segmentation après égalisation", color_continuous_scale='gray').show()

## 2 Morphologie mathématique : granulométrie des globules

In [76]:
kernel_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
im_clean = cv2.morphologyEx(im_bin, cv2.MORPH_OPEN, kernel_small)

In [77]:
kernel_sep = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
im_sep = cv2.erode(im_clean, kernel_sep)

In [ ]:
def reconstruction(marker, mask):
    kernel = np.ones((3, 3), np.uint8)
    
    reconstructed = marker.copy()
    
    while True:
        previous = reconstructed.copy()
        dilation = cv2.dilate(reconstructed, kernel)
        reconstructed = np.minimum(dilation, mask)
        if np.array_equal(reconstructed, previous):
            break
            
    return reconstructed

In [87]:
marker = np.zeros_like(im_sep)
marker[0,:] = im_sep[0,:]
marker[-1,:] = im_sep[-1,:]
marker[:,0] = im_sep[:,0]
marker[:,-1] = im_sep[:,-1]
objets_bords = reconstruction(marker, im_sep)

px.imshow(im_sep, title="Globules complets (nettoyés)", color_continuous_scale='gray').show()

im_final = cv2.subtract(im_sep, objets_bords)
px.imshow(im_final, title="Globules complets (nettoyés)", color_continuous_scale='gray').show()

In [81]:
radii = np.arange(1, 60, 2)
counts = []

for r in radii:
    se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (r, r))
    im_opened = cv2.morphologyEx(im_final, cv2.MORPH_OPEN, se)
    n_obj = tsi.bweuler(im_opened)
    counts.append(n_obj)

granulo_curve = -np.diff(counts)

In [82]:
fig_counts = px.line(x=radii, y=counts, labels={'x': 'Rayon de l\'élément structurant', 'y': 'Nombre d\'objets restants'}, title="Décroissance du nombre d'objets (Tamisage)")
fig_counts.show()

In [84]:
centers_granulo = radii[:-1] 
H = granulo_curve
Hc = np.cumsum(H)            

tsi.plotHistograms(centers_granulo, H, Hc, title="Courbe Granulométrique des globules")